# TabPFN → drzewo decyzyjne (rozwiązanie 2)

Colab: **Runtime → Change runtime type → T4 GPU**, potem Run all.

TabPFN uczy się z `val.csv`, etykietuje `train.csv`, a sklearn-drzewo destyluje te decyzje na nazwanych sygnaturach akustycznych. Aplikacja warsztatowa (`app_tabpfn.py`) liczy już tylko drzewo — CPU, ścieżka if/then, punkty za explainability.

In [ ]:
import os
from pathlib import Path

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB:
    %pip install -q tabpfn scikit-learn pandas numpy torch joblib plotly

PROJECT_DIR = Path.cwd()
required_files = ["val.csv", "train.csv", "test.csv", "tabpfn_diagnose.py"]
missing_files = [name for name in required_files if not (PROJECT_DIR / name).exists()]
print("Środowisko:", "Colab" if IN_COLAB else "lokalne")
print("Katalog roboczy:", PROJECT_DIR)

Wybierz lokalne pliki: val.csv, train.csv, test.csv oraz tabpfn_diagnose.py


In [ ]:
if IN_COLAB and missing_files:
    from google.colab import files
    print("Kliknij Run tej komórki, a następnie wybierz: val.csv, train.csv, test.csv oraz tabpfn_diagnose.py")
    files.upload()
    missing_files = [name for name in required_files if not (PROJECT_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        f"Brak plików w katalogu {PROJECT_DIR}: {', '.join(missing_files)}"
    )

print("Pliki znalezione:", ", ".join(required_files))

In [ ]:
import sys

sys.path.insert(0, str(PROJECT_DIR))
from tabpfn_diagnose import TabPFNTreeDiagnoser, pick_device
import pandas as pd

val = pd.read_csv(PROJECT_DIR / "val.csv")
train = pd.read_csv(PROJECT_DIR / "train.csv")
test = pd.read_csv(PROJECT_DIR / "test.csv")
print(val.shape, train.shape, test.shape, "device=", pick_device())

## Nauczyciel + destylacja

Na T4 włącz `--cv` w komórce poniżej (GroupKFold po silniku). Na CPU zostaw `do_cv=False`.

In [ ]:
device = pick_device()
n_estimators = 8 if device == "cuda" else 4
do_cv = device == "cuda"  # T4: uczciwy score; CPU: pomiń

model = TabPFNTreeDiagnoser().fit(
    val,
    train,
    device=device,
    n_estimators=n_estimators,
    do_cv=do_cv,
)
model.save()
print(model.meta)

## Drzewo, które zobaczy mechanik

In [ ]:
print(model.rules_text())

## Submit + zgodność nauczyciel / student

In [ ]:
sub_tree = model.predict(test)
sub_tabpfn = model.predict_teacher(test, model.teacher_)
sub_tree.to_csv(PROJECT_DIR / "predictions_tree.csv", index=False)
sub_tabpfn.to_csv(PROJECT_DIR / "predictions_tabpfn.csv", index=False)
agree = (sub_tree["label"] == sub_tabpfn["label"]).mean()
print(f"zgoda drzewo vs TabPFN na teście: {agree:.3f}")
print("TabPFN\n", sub_tabpfn["label"].value_counts())
print("drzewo\n", sub_tree["label"].value_counts())

Lokalnie po pobraniu `diagnoser_tree.joblib` do `artifacts/`:

```bash
streamlit run app_tabpfn.py
```